# TX 台指期近月 回測模板

## 策略規則摘要

| 項目 | 說明 |
|------|------|
| **商品** | 台股指數近月(一般)(FITX*1.TF) |
| **訊號** | _(在此描述你的訊號邏輯)_ |
| **進場條件** | signal == +1（多）或 -1（空） |
| **出場條件** | 訊號消失、反轉、停損或停利 |
| **部位方向** | 多空雙向，固定 1 口 |
| **每日部位歸零** | 否 |
| **模擬逐筆洗價** | 否 |
| **交易費用** | 期貨 105 元/口 + 滑價 1 檔（200 元），來回 610 元 |




In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
import time
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# 設定 matplotlib 中文字型
for font_name in ['Noto Sans CJK TC', 'Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']:
    try:
        matplotlib.font_manager.findfont(font_name, fallback_to_default=False)
        plt.rcParams['font.sans-serif'] = [font_name, 'DejaVu Sans']
        break
    except Exception:
        continue
plt.rcParams['axes.unicode_minus'] = False

# 檢查 plotly 是否可用
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    from pathlib import Path
    HAS_PLOTLY = True
    print("Plotly 可用")
except ImportError:
    HAS_PLOTLY = False
    print("Plotly 不可用，將使用 matplotlib 替代")

# import finlab
# finlab.login('gBRGOIBZ4otUfVf5nLkBso0tfxylvwpCZQTUlgF+sUqbKi1HhdqjgCdseTnRSYOI#vip_m')
# from finlab import data



In [ ]:
import glob
import os

# Data Loading
FUTURE_DIR = "./KBar_1m_ML/KBar_1m_ML"

csv_files = sorted(glob.glob(os.path.join(FUTURE_DIR, "*.csv")))
print(f"找到 {len(csv_files)} 個 CSV 檔案")

df = pd.concat(
    [pd.read_csv(f, encoding="utf-8-sig") for f in csv_files],
    ignore_index=True
)

df["datetime"] = pd.to_datetime(df["datetime"])
df = df.set_index("datetime").sort_index()

print(f"資料筆數：{len(df):,}")
print(f"期間：{df.index[0]} ~ {df.index[-1]}")
print(f"欄位：{list(df.columns)}")
print(f"\n日盤筆數：{(df['session']=='day').sum():,}")
print(f"夜盤筆數：{(df['session']=='night').sum():,}")
df.head(3)


In [ ]:

BENCHMARK_EXCEL = "./數k棒重力流-非多即空20260416.xlsx"
df_benchmark = pd.read_excel(BENCHMARK_EXCEL, sheet_name="交易分析")
df_benchmark["進場時間"] = pd.to_datetime(df_benchmark["進場時間"], errors="coerce")
df_benchmark["出場時間"] = pd.to_datetime(df_benchmark["出場時間"], errors="coerce")

print(f"基準資料筆數：{len(df_benchmark):,}")
print(f"期間：{df_benchmark['進場時間'].min()} ~ {df_benchmark['出場時間'].max()}")
print(f"欄位：{list(df_benchmark.columns)}")
df_benchmark.head(3)

# benchmark 累計獲利線
benchmark = df_benchmark.copy()
benchmark["時間"] = benchmark["出場時間"].fillna(benchmark["進場時間"])
benchmark["累計獲利金額"] = pd.to_numeric(
    benchmark["累計獲利金額"].astype(str).str.replace(",", ""),
    errors="coerce"
)

benchmark = (
    benchmark.dropna(subset=["時間", "累計獲利金額"])
    .sort_values("時間")
    .drop_duplicates(subset=["時間"], keep="last")
    .set_index("時間")["累計獲利金額"]
    .astype(float)
)

# 轉成「權益曲線」並補上 plot_report 需要的 init_capital 屬性
init_capital = getattr(globals().get("cfg", None), "init_capital", 1_000_000)
benchmark = benchmark + init_capital
benchmark.name = "benchmark_equity"
benchmark.init_capital = init_capital



In [ ]:
import pandas as pd
import numpy as np
import datetime

# Load data
# df = pd.read_parquet("data.parquet")

# 1. Calculate Rolling Stats
# Shift by 1 to avoid lookahead bias
# 20-bar lookback for low/high/median
low_20_min = df['low'].rolling(20).min().shift(1)
close_20_med = df['close'].rolling(20).median().shift(1)
high_20_max = df['high'].rolling(20).max().shift(1)

# 2. Compute Lower and Upper Bounds
# lower = 0.7 * rolling_low + 0.3 * rolling_median
lower = 0.7 * low_20_min + 0.3 * close_20_med
# upper = rolling_high
upper = high_20_max

# 3. Percentile Position
# pct = (close - lower) / (upper - lower)
diff = upper - lower
# Avoid division by zero
diff_safe = diff.replace(0, np.nan)
pct = (df['close'] - lower) / diff_safe
# Clip to [0, 1]
pct = pct.clip(0, 1)

# 4. Base Signal
# Centered around 0.5 -> [-1, 1]
centered = (pct - 0.5) * 2
# Tanh activation to bound the signal
base = np.tanh(centered.clip(-3, 3))

# 5. Skewness Regime Filter
# 60-bar lookback for skewness
ret = df['close'].pct_change()
skew = ret.rolling(60).skew()
# Regime filter: exp(-|skew| / 0.5)
# High skew (trending) -> low regime value (suppress signal)
# Low skew (ranging) -> high regime value (amplify signal)
regime = np.exp(-np.abs(skew) / 0.5)

# 6. Volatility Scaling
# 60-bar vol, normalized by 200-bar mean vol
vol = ret.rolling(60).std()
vol_mean = vol.rolling(200).mean()
# Avoid div by zero
vol_norm = vol / vol_mean.replace(0, np.nan)
# Clip scale to [0.6, 1.8]
scale = vol_norm.clip(0.6, 1.8)

# 7. Combine Raw Signal
# modulation = regime * scale
modulation = regime * scale
raw_signal = base * modulation

# 8. One-Shot Logic
# Fire once at day-session open
is_new_day = (df['session'] == 'day') & (df['session'].shift(1) != 'day')
# Apply where and forward fill
signal = raw_signal.where(is_new_day).ffill().fillna(0)

# 9. Time Mask
# Only trade 09:15 - 13:00 in day session
# Calculate minutes from midnight for vectorized comparison
minutes = df.index.hour * 60 + df.index.minute
start_min = 9 * 60 + 15  # 555
end_min = 13 * 60        # 780

time_cond = (minutes >= start_min) & (minutes <= end_min)
# Combine with session filter
time_mask = (df['session'] == 'day') & time_cond

# Apply mask
signal = signal * time_mask.astype(float)

# 10. Final Validation and Save
if isinstance(signal, pd.DataFrame):
    signal = signal.squeeze()
assert isinstance(signal, pd.Series), (f"signal must be pd.Series before saving, got {type(signal)}")
signal = signal.fillna(0)
signal.name = "signal"
# signal.to_hdf("result.h5", key="signal", mode="w")

In [ ]:
# Print signal stats
print(f"Signal stats:")
print(f"  Mean: {signal.mean():.4f}")
print(f"  Std: {signal.std():.4f}")
print(f"  Min: {signal.min():.4f}")
print(f"  Max: {signal.max():.4f}")

LONG_THRESHOLD = 0
SHORT_THRESHOLD = -0
signal = signal.apply(lambda x: 1 if x > LONG_THRESHOLD else (-1 if x < SHORT_THRESHOLD else 0))

# find unique values in signal
print(f"Unique signal values: {signal.unique()}")

# Analyze signal distribution
signal_counts = signal.value_counts().sort_index()
print("\nSignal distribution:")
print(signal_counts)

## 訊號產生與統計

In [ ]:
# ── Signal 視覺化：進出場標記 ──────────────────────────────────────
PLOT_START = "2026-03-01"
PLOT_END   = "2026-03-31"
DAY_ONLY   = False   # True = 只看日盤，False = 全時段

# ── 進出場邊緣偵測
entry_long  = signal[(signal ==  1) & (signal.shift(1, fill_value=0) !=  1)]
entry_short = signal[(signal == -1) & (signal.shift(1, fill_value=0) != -1)]
exit_long   = signal[(signal !=  1) & (signal.shift(1, fill_value=0) ==  1)]
exit_short  = signal[(signal != -1) & (signal.shift(1, fill_value=0) == -1)]


# ── 期間切片
plot_df  = df.loc[PLOT_START:PLOT_END].copy()
if DAY_ONLY:
    plot_df = plot_df[plot_df["session"] == "day"]
plot_sig = signal.reindex(plot_df.index).fillna(0)

def _clip(s):
    return s[s.index.isin(plot_df.index)]

el, es, xl, xs = _clip(entry_long), _clip(entry_short), _clip(exit_long), _clip(exit_short)
print(f"{'日盤' if DAY_ONLY else '全時段'}  {PLOT_START} ~ {PLOT_END}  |  "
      f"{len(plot_df):,} 根 K棒  |  "
      f"多進 {len(el)}  空進 {len(es)}  多出 {len(xl)}  空出 {len(xs)}")

# ── 把連續相同 signal 合成 span（避免逐 bar 加 vrect）
def signal_spans(sig, val):
    spans = []
    in_span, start = False, None
    for ts, v in sig.items():
        if v == val and not in_span:
            in_span, start = True, ts
        elif v != val and in_span:
            spans.append((start, ts))
            in_span = False
    if in_span:
        spans.append((start, sig.index[-1]))
    return spans

long_spans  = signal_spans(plot_sig,  1)
short_spans = signal_spans(plot_sig, -1)

# ── 20 MA
ma20 = plot_df["close"].rolling(20, min_periods=1).mean()

# ────────────────── Plotly ──────────────────
if HAS_PLOTLY:
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.62, 0.20, 0.18],
        vertical_spacing=0.02,
    )

    # ── 蠟燭圖
    fig.add_trace(go.Candlestick(
        x=plot_df.index,
        open=plot_df["open"], high=plot_df["high"],
        low=plot_df["low"],   close=plot_df["close"],
        name="K棒",
        increasing=dict(line=dict(color="#ef5350", width=1), fillcolor="#ef5350"),
        decreasing=dict(line=dict(color="#26a69a", width=1), fillcolor="#26a69a"),
        showlegend=False,
    ), row=1, col=1)

    # 拿掉日與日之間的空白
    fig.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"]),                 # 週末
            dict(bounds=[13.75, 8.75], pattern="hour"),  # 13:45 ~ 隔日 08:45
        ]
    )

    # ── MA20
    fig.add_trace(go.Scatter(
        x=plot_df.index, y=ma20,
        name="MA20", line=dict(color="#f39c12", width=1.2, dash="dot"),
        opacity=0.85,
    ), row=1, col=1)

    # ── signal 色塊（span 合併版）
    for s, e in long_spans:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(46,204,113,0.12)",
                      layer="below", line_width=0)
    for s, e in short_spans:
        fig.add_vrect(x0=s, x1=e, fillcolor="rgba(231,76,60,0.12)",
                      layer="below", line_width=0)

    # ── 多單進場 ▲
    if len(el):
        fig.add_trace(go.Scatter(
            x=el.index, y=plot_df.loc[el.index, "low"] ,
            mode="markers", name="多進",
            marker=dict(symbol="triangle-up", size=14, color="#2ecc71",
                        line=dict(width=1.5, color="#145a32")),
        ), row=1, col=1)

    # ── 空單進場 ▼
    if len(es):
        fig.add_trace(go.Scatter(
            x=es.index, y=plot_df.loc[es.index, "high"],
            mode="markers", name="空進",
            marker=dict(symbol="triangle-down", size=14, color="#e74c3c",
                        line=dict(width=1.5, color="#7b241c")),
        ), row=1, col=1)

    # ── 多單出場 ×
    if len(xl):
        fig.add_trace(go.Scatter(
            x=xl.index, y=plot_df.loc[xl.index, "close"],
            mode="markers", name="多出",
            marker=dict(symbol="x", size=11, color="#27ae60",
                        line=dict(width=2.5)),
        ), row=1, col=1)

    # ── 空單出場 ×
    if len(xs):
        fig.add_trace(go.Scatter(
            x=xs.index, y=plot_df.loc[xs.index, "close"],
            mode="markers", name="空出",
            marker=dict(symbol="x", size=11, color="#c0392b",
                        line=dict(width=2.5)),
        ), row=1, col=1)

    # ── Volume
    vol_color = ["rgba(239,83,80,0.7)" if c >= o else "rgba(38,166,154,0.7)"
                 for c, o in zip(plot_df["close"], plot_df["open"])]
    fig.add_trace(go.Bar(
        x=plot_df.index, y=plot_df["volume"],
        name="Volume", marker_color=vol_color, showlegend=False,
    ), row=2, col=1)

    # ── Signal
    fig.add_trace(go.Scatter(
        x=plot_sig.index, y=plot_sig.values,
        name="Signal", mode="lines",
        line=dict(color="#8e44ad", width=1.5),
        fill="tozeroy",
        fillcolor="rgba(142,68,173,0.08)",
        showlegend=False,
    ), row=3, col=1)
    fig.add_hline(y=0, line_color="rgba(0,0,0,0.3)", line_dash="dot", row=3, col=1)

    fig.update_layout(
        template="plotly_white",
        height=780,
        title=dict(text=f"<b>TX 1分K  進出場標記</b>　{PLOT_START} ～ {PLOT_END}",
                   font=dict(size=16)),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.03, x=1, xanchor="right",
                    bgcolor="rgba(255,255,255,0.8)", borderwidth=1),
        margin=dict(l=60, r=30, t=70, b=40),
        plot_bgcolor="#fafafa",
        paper_bgcolor="white",
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.06)", tickangle=-30)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.06)")
    fig.update_yaxes(title_text="價格", row=1, col=1)
    fig.update_yaxes(title_text="量", row=2, col=1)
    fig.update_yaxes(tickvals=[-1, 0, 1], ticktext=["空", "—", "多"],
                     title_text="訊號", row=3, col=1)
    fig.show()

# ────────────────── Matplotlib 備案 ──────────────────
else:
    from matplotlib.patches import FancyArrowPatch
    fig, axes = plt.subplots(3, 1, figsize=(20, 10), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1, 1]},
                             facecolor="white")
    fig.subplots_adjust(hspace=0.05)

    ax = axes[0]
    ax.set_facecolor("#fafafa")
    for s, e in long_spans:
        ax.axvspan(s, e, color="#2ecc71", alpha=0.1, lw=0)
    for s, e in short_spans:
        ax.axvspan(s, e, color="#e74c3c", alpha=0.1, lw=0)
    ax.plot(plot_df.index, plot_df["close"], color="#2c3e50", linewidth=0.9, label="Close")
    ax.plot(plot_df.index, ma20, color="#f39c12", linewidth=1, linestyle="--", label="MA20", alpha=0.8)
    std = plot_df["close"].std()
    if len(el): ax.scatter(el.index, plot_df.loc[el.index,"low"] - std*0.3,
                           marker="^", s=100, color="#2ecc71", zorder=5, label="多進", edgecolors="#145a32", linewidths=1)
    if len(es): ax.scatter(es.index, plot_df.loc[es.index,"high"] + std*0.3,
                           marker="v", s=100, color="#e74c3c", zorder=5, label="空進", edgecolors="#7b241c", linewidths=1)
    if len(xl): ax.scatter(xl.index, plot_df.loc[xl.index,"close"],
                           marker="x", s=90, color="#27ae60", zorder=5, label="多出", linewidths=2)
    if len(xs): ax.scatter(xs.index, plot_df.loc[xs.index,"close"],
                           marker="x", s=90, color="#c0392b", zorder=5, label="空出", linewidths=2)
    ax.set_ylabel("價格", fontsize=11)
    ax.legend(loc="upper left", fontsize=9, framealpha=0.8)
    ax.set_title(f"TX 1分K  進出場標記　{PLOT_START} ～ {PLOT_END}", fontsize=13, pad=10)
    ax.grid(axis="y", color="gray", alpha=0.2)

    axes[1].set_facecolor("#fafafa")
    vol_c = ["#ef5350" if c >= o else "#26a69a"
             for c, o in zip(plot_df["close"], plot_df["open"])]
    axes[1].bar(plot_df.index, plot_df["volume"], color=vol_c, width=0.0004, alpha=0.8)
    axes[1].set_ylabel("量", fontsize=11)
    axes[1].grid(axis="y", color="gray", alpha=0.2)

    axes[2].set_facecolor("#fafafa")
    axes[2].plot(plot_sig.index, plot_sig.values, color="#8e44ad", linewidth=1.2)
    axes[2].fill_between(plot_sig.index, plot_sig.values, 0,
                         where=plot_sig > 0, color="#2ecc71", alpha=0.2)
    axes[2].fill_between(plot_sig.index, plot_sig.values, 0,
                         where=plot_sig < 0, color="#e74c3c", alpha=0.2)
    axes[2].axhline(0, color="gray", linestyle=":", linewidth=0.8)
    axes[2].set_yticks([-1, 0, 1])
    axes[2].set_yticklabels(["空", "—", "多"], fontsize=10)
    axes[2].set_ylabel("訊號", fontsize=11)
    axes[2].grid(axis="y", color="gray", alpha=0.2)

    plt.tight_layout()
    plt.savefig("signal_chart.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("已存檔 signal_chart.png")

## 回測引擎

**執行邏輯：**
- 訊號邊緣偵測（signal 從 0 → ±1）→ **下一根開盤**成交
- 多空反轉：同一根開盤同時平舊倉、開新倉
- 停損 / 停利 / 最大持倉根數：**收盤價**評估，**下一根開盤**執行
- 成本：手續費 105 TWD + 滑價 1 檔（200 TWD）= **單邊 305 TWD；來回 610 TWD**


In [ ]:
# import sys, os, time
# sys.path.insert(0, '/home/intern2/workspace/intern-dev/strategy')
from backtest_engiene import BacktestConfig, run_backtest, print_report

# ─── 回測參數（對應 XQ/MultiCharts 設定）────────────────────────────────
# 商品    ：台股指數近月(一般)(FITX*1.TF)
# 執行頻率：1 分鐘
# 費  用  ：期貨 105 元/口  + 滑價 1 檔（200 TWD）= 單邊 305 TWD
# 進場價  ：市價 +1 檔（買）/ 市價 -1 檔（賣）→ 以 next_open 近似
# 每日歸零：否　　　逐筆洗價：否

START_DATE = "2020-01-01"
END_DATE   = "2026-04-15"

cfg = BacktestConfig(
    point_value        = 200,          # 大台每點 200 TWD
    fee_per_side       = 105,          # 手續費 105 元/口
    slippage_ticks     = 1,            # 滑價 1 檔 = 200 TWD/口
    tick_value         = 200,
    contracts          = 1,
    execution          = "next_open",  # 下一根開盤成交（最保守）
    stop_loss_points   = None,         # 停損點數（None = 不啟用）
    take_profit_points = None,         # 停利點數（None = 不啟用）
    max_hold_bars      = None,         # 最大持倉根數（None = 不限）
    session_filter     = None,         # None = 日夜全時段
    init_capital       = 1_000_000,
)

print(cfg)
print(f"\n來回成本：{cfg.cost_roundtrip:.0f} TWD/口")
print(f"  手續費：{cfg.fee_per_side:.0f} × 2 = {cfg.fee_per_side * 2:.0f} TWD")
print(f"  滑  價：1 檔 × {cfg.slippage_twd:.0f} × 2 = {cfg.slippage_twd * 2:.0f} TWD")

# ─── 日期篩選 ─────────────────────────────────────────────────────────────
bt_df  = df.loc[START_DATE:END_DATE].copy()
bt_sig = signal.loc[START_DATE:END_DATE].copy()

print(f"\n回測期間：{bt_df.index[0]} ~ {bt_df.index[-1]}")
print(f"K棒數量：{len(bt_df):,}　訊號（非零）：{(bt_sig != 0).sum():,}")

# ─── 執行回測 ─────────────────────────────────────────────────────────────
t0 = time.time()
trades_df, equity = run_backtest(bt_df, bt_sig, cfg)
elapsed = time.time() - t0

print(f"\n回測完成，耗時 {elapsed:.2f} 秒")
print_report(trades_df, equity, cfg)


## 績效統計

In [ ]:
from backtest_engiene import calc_metrics

m = calc_metrics(trades_df, equity, cfg)

# ── 績效摘要表
perf_table = pd.DataFrame({
    '指標': ['總交易筆數', '勝率', '平均獲利', '平均虧損', '盈虧比',
             '獲利因子', '累計損益', 'CAGR', 'Sharpe', '最大回撤', 'Calmar'],
    '數值': [
        f"{m['total_trades']:,} 筆",
        f"{m['win_rate']:.1f}%",
        f"{m['avg_win_twd']:,.0f} TWD",
        f"{m['avg_loss_twd']:,.0f} TWD",
        f"{m['payoff_ratio']:.2f}",
        f"{m['profit_factor']:.3f}",
        f"{m['total_pnl_twd']:+,.0f} TWD",
        f"{m['cagr'] * 100:+.2f}%",
        f"{m['sharpe']:.3f}",
        f"{m['max_drawdown'] * 100:.2f}%",
        f"{m['calmar']:.3f}",
    ],
}).set_index('指標')

print("績效摘要：")
display(perf_table)

# ── 年度損益表
annual = (
    trades_df.copy()
    .assign(year=lambda x: x['exit_time'].dt.year)
    .groupby('year')['pnl_twd']
    .agg(['sum', 'count', lambda x: (x > 0).sum() / len(x) * 100])
    .rename(columns={'sum': '淨損益(TWD)', 'count': '筆數', '<lambda_0>': '勝率(%)'})
)
annual_fmt = annual.copy()
annual_fmt['淨損益(TWD)'] = annual['淨損益(TWD)'].apply(lambda x: f"{x:+,.0f}")
annual_fmt['勝率(%)']    = annual['勝率(%)'].apply(lambda x: f"{x:.1f}%")
annual_fmt.index.name = '年度'
print("\n年度損益：")
display(annual_fmt)

# ── 多空分析表
rows = []
for direction in ['多', '空', '合計']:
    sub = trades_df if direction == '合計' else trades_df[trades_df['direction'] == direction]
    if len(sub) == 0:
        continue
    pnl  = sub['pnl_twd']
    wins = pnl[pnl > 0]
    loss = pnl[pnl <= 0]
    pf   = wins.sum() / abs(loss.sum()) if loss.sum() != 0 else float('inf')
    rows.append({
        '方向': direction,
        '筆數': f"{len(sub):,}",
        '勝率': f"{len(wins) / len(sub) * 100:.1f}%",
        '平均獲利 (TWD)': f"{wins.mean():,.0f}" if len(wins) else "—",
        '平均虧損 (TWD)': f"{loss.mean():,.0f}" if len(loss) else "—",
        'PF': f"{pf:.2f}",
        '累計損益 (TWD)': f"{pnl.sum():+,.0f}",
    })
print("\n多空分析：")
display(pd.DataFrame(rows).set_index('方向'))


## 視覺化

In [ ]:
from backtest_engiene import plot_report
plot_report(bt_df, trades_df, benchmark, equity, cfg, title="TX 台指期近月 回測結果")


## 交易明細

In [ ]:
# ─── 交易明細 ─────────────────────────────────────────────────────────────
display_cols = ['entry_time', 'entry_price', 'exit_time', 'exit_price',
                'direction', 'hold_bars', 'pnl_points', 'pnl_twd', 'exit_reason']
fmt = {'entry_price': '{:.0f}', 'exit_price': '{:.0f}',
       'pnl_points': '{:+.0f}', 'pnl_twd': '{:+,.0f}'}

print(f"交易總筆數：{len(trades_df):,}\n")

print("前 20 筆交易：")
display(trades_df[display_cols].head(20).style.format(fmt))

print("\nTop 20 獲利交易：")
display(trades_df[display_cols].nlargest(20, 'pnl_twd').style.format(fmt))

print("\nTop 20 虧損交易：")
display(trades_df[display_cols].nsmallest(20, 'pnl_twd').style.format(fmt))

print("\nTop 10 2020虧損交易：")
display(trades_df[trades_df['entry_time'].dt.year == 2020][display_cols].nsmallest(10, 'pnl_twd').style.format(fmt))

print("\nTop 10 2023虧損交易：")
display(trades_df[trades_df['entry_time'].dt.year == 2023][display_cols].nsmallest(10, 'pnl_twd').style.format(fmt))

## 時間結構分析（Profit Factor 熱力圖）

In [ ]:
# ─── Profit Factor 函數 ───────────────────────────────────────────────────
def profit_factor(pnl):
    gains  = pnl[pnl > 0].sum()
    losses = abs(pnl[pnl <= 0].sum())
    if losses == 0:
        return float('inf') if gains > 0 else 0.0
    return gains / losses

# ─── 準備時間特徵 ─────────────────────────────────────────────────────────
trades_time = trades_df.copy()
trades_time['exit_weekday'] = trades_time['exit_time'].dt.weekday
trades_time['exit_week']    = trades_time['exit_time'].dt.isocalendar().week.astype(int)
trades_time['exit_month']   = trades_time['exit_time'].dt.month
trades_time['exit_year']    = trades_time['exit_time'].dt.year
trades_time['exit_wom']     = (trades_time['exit_time'].dt.day - 1) // 7 + 1

# ─── 熱力圖 1：ISO Week × Weekday ─────────────────────────────────────────
pf_ww = (
    trades_time.groupby(['exit_week', 'exit_weekday'])['pnl_twd']
    .apply(profit_factor)
    .unstack()
    .clip(0, 3)
)
pf_ww.columns = ['週一', '週二', '週三', '週四', '週五'][:pf_ww.shape[1]]

if HAS_PLOTLY:
    fig_hm1 = go.Figure(go.Heatmap(
        z=pf_ww.values, x=pf_ww.columns, y=pf_ww.index,
        colorscale='RdYlGn', zmid=1.0, colorbar=dict(title='PF'),
    ))
    fig_hm1.update_layout(
        template='plotly_white', height=700,
        title='Profit Factor 熱力圖（ISO 週 × 星期）',
        xaxis_title='星期', yaxis_title='ISO 週',
    )
    fig_hm1.show()
else:
    fig, ax = plt.subplots(figsize=(8, 16))
    im = ax.imshow(pf_ww.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=3)
    ax.set_xticks(range(len(pf_ww.columns))); ax.set_xticklabels(pf_ww.columns)
    ax.set_yticks(range(len(pf_ww.index))); ax.set_yticklabels(pf_ww.index)
    ax.set_title('Profit Factor 熱力圖（ISO 週 × 星期）')
    plt.colorbar(im, ax=ax, label='PF'); plt.tight_layout(); plt.show()

# ─── 熱力圖 2：年月 × 月內週數 ────────────────────────────────────────────
trades_time['ym'] = trades_time['exit_time'].dt.to_period('M').astype(str)
pf_ym = (
    trades_time.groupby(['ym', 'exit_wom'])['pnl_twd']
    .apply(profit_factor)
    .unstack()
    .clip(0, 3)
)
pf_ym.columns = [f'第{int(c)}週' for c in pf_ym.columns]

if HAS_PLOTLY:
    fig_hm2 = go.Figure(go.Heatmap(
        z=pf_ym.values, x=pf_ym.columns, y=pf_ym.index,
        colorscale='RdYlGn', zmid=1.0, colorbar=dict(title='PF'),
    ))
    fig_hm2.update_layout(
        template='plotly_white', height=800,
        title='Profit Factor 熱力圖（年月 × 月內週數）',
        xaxis_title='月內週數', yaxis_title='年月',
    )
    fig_hm2.show()
else:
    fig, ax = plt.subplots(figsize=(8, 20))
    im = ax.imshow(pf_ym.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=3)
    ax.set_xticks(range(len(pf_ym.columns))); ax.set_xticklabels(pf_ym.columns)
    ax.set_yticks(range(0, len(pf_ym.index), 6)); ax.set_yticklabels(pf_ym.index[::6])
    ax.set_title('Profit Factor 熱力圖（年月 × 月內週數）')
    plt.colorbar(im, ax=ax, label='PF'); plt.tight_layout(); plt.show()


In [ ]:
# ─── 星期別中位數損益 ─────────────────────────────────────────────────────
weekday_names = ['週一', '週二', '週三', '週四', '週五']
weekday_med = trades_time.groupby('exit_weekday')['pnl_twd'].median()
weekday_med.index = [weekday_names[i] for i in weekday_med.index if i < 5]

# ─── 月內週別中位數損益 ───────────────────────────────────────────────────
wom_med = trades_time.groupby('exit_wom')['pnl_twd'].median()
wom_med.index = [f'第{int(w)}週' for w in wom_med.index]

if HAS_PLOTLY:
    fig_wd = go.Figure(go.Bar(
        x=weekday_med.index, y=weekday_med.values,
        marker_color=['#e74c3c' if v < 0 else '#2ecc71' for v in weekday_med.values],
    ))
    fig_wd.add_hline(y=0, line_dash='dot', line_color='gray')
    fig_wd.update_layout(template='plotly_white', title='星期別中位數損益',
                         yaxis_title='中位數損益 (TWD)', height=350)
    fig_wd.show()

    fig_wom = go.Figure(go.Bar(
        x=wom_med.index, y=wom_med.values,
        marker_color=['#e74c3c' if v < 0 else '#2ecc71' for v in wom_med.values],
    ))
    fig_wom.add_hline(y=0, line_dash='dot', line_color='gray')
    fig_wom.update_layout(template='plotly_white', title='月內週別中位數損益',
                          yaxis_title='中位數損益 (TWD)', height=350)
    fig_wom.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, med, title in [(axes[0], weekday_med, '星期別中位數損益'),
                           (axes[1], wom_med, '月內週別中位數損益')]:
        colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in med.values]
        ax.bar(med.index, med.values, color=colors)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_title(title); ax.set_ylabel('中位數損益 (TWD)')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout(); plt.show()

print("\n回測分析完成！")


In [ ]:
# ============================================
# 交易時間特徵績效分析
# ============================================

def profit_factor(pnl):
    gains = pnl[pnl > 0].sum()
    losses = abs(pnl[pnl <= 0].sum())
    if losses == 0:
        return float('inf') if gains > 0 else 0.0
    return gains / losses

def time_performance_table(pnl_series, name=""):
    """產出標準化時間績效表格"""
    wins = pnl_series[pnl_series > 0]
    loss = pnl_series[pnl_series <= 0]
    pf = profit_factor(pnl_series)
    return {
        '樣本數': len(pnl_series),
        '勝率(%)': f"{len(wins)/len(pnl_series)*100:.1f}" if len(pnl_series) > 0 else "—",
        '獲利因子': f"{pf:.2f}" if pf != float('inf') else "∞",
        '總損益': f"{pnl_series.sum():+,.0f}",
        '平均損益': f"{pnl_series.mean():+,.0f}",
        '中位數損益': f"{pnl_series.median():+,.0f}",
        '平均獲利': f"{wins.mean():,.0f}" if len(wins) > 0 else "—",
        '平均虧損': f"{loss.mean():,.0f}" if len(loss) > 0 else "—",
    }

# 複製交易資料並萃取時間特徵
tt = trades_df.copy()
tt['entry_hour']   = tt['entry_time'].dt.hour
tt['entry_minute'] = tt['entry_time'].dt.minute
tt['entry_time_val'] = tt['entry_time'].dt.hour + tt['entry_time'].dt.minute / 60
tt['exit_hour']    = tt['exit_time'].dt.hour
tt['exit_minute']  = tt['exit_time'].dt.minute
tt['exit_time_val']  = tt['exit_time'].dt.hour + tt['exit_time'].dt.minute / 60
tt['entry_weekday'] = tt['entry_time'].dt.weekday
tt['exit_weekday']  = tt['exit_time'].dt.weekday
tt['entry_month']   = tt['entry_time'].dt.month
tt['exit_month']    = tt['exit_time'].dt.month

# 日盤時段分類（更細緻）
def classify_day_session(t):
    """t 為 datetime.time 或 hour+minute"""
    if hasattr(t, 'hour'):
        h, m = t.hour, t.minute
    else:
        h, m = int(t), int((t % 1) * 60)
    if (h == 8 and m >= 45) or (h == 9 and m < 30):
        return '08:45-09:30 開盤初期'
    elif h == 9 and m >= 30 or h == 10:
        return '09:30-10:59 早盤'
    elif h == 11 or (h == 12 and m == 0):
        return '11:00-11:45 午盤收尾'
    else:
        return '其他'

tt['entry_session_seg'] = tt['entry_time'].apply(classify_day_session)
tt['exit_session_seg']  = tt['exit_time'].apply(classify_day_session)

weekday_names = {0:'週一', 1:'週二', 2:'週三', 3:'週四', 4:'週五', 5:'週六', 6:'週日'}

print("=" * 70)
print("一、進場時間特徵分析")
print("=" * 70)

# ── 1.1 進場15分鐘別績效 ──
print("\n【1.1 進場15分鐘別績效】")
entry_15min_stats = []
for h in sorted(tt['entry_hour'].unique()):
    for m in [0, 15, 30, 45]:
        sub = tt[(tt['entry_hour'] == h) & (tt['entry_minute'] // 15 == m // 15)]['pnl_twd']
        if len(sub) >= 3:  # 樣本數過少的時間段不列入分析
            stats = time_performance_table(sub, f"{h:02d}:{m:02d}")
            stats['進場時間'] = f"{h:02d}:{m:02d}"
            entry_15min_stats.append(stats)
entry_15min_df = pd.DataFrame(entry_15min_stats).set_index('進場時間')
display(entry_15min_df)

# ── 1.2 進場時段（細分）績效 ──
print("\n【1.2 進場時段（日盤細分）績效】")
# seg_stats = []
# for seg in ['08:45-09:30 開盤初期', '09:30-10:59 早盤', '11:00-11:45 午盤收尾']:
#     sub = tt[tt['entry_session_seg'] == seg]['pnl_twd']
#     if len(sub) > 0:
#         stats = time_performance_table(sub, seg)
#         stats['進場時段'] = seg
#         seg_stats.append(stats)
# seg_df = pd.DataFrame(seg_stats).set_index('進場時段')
# display(seg_df)

# ── 1.3 進場星期績效 ──
print("\n【1.3 進場星期績效】")
wd_stats = []
for wd in range(5):  # 週一到週五
    sub = tt[tt['entry_weekday'] == wd]['pnl_twd']
    if len(sub) > 0:
        stats = time_performance_table(sub, weekday_names[wd])
        stats['星期'] = weekday_names[wd]
        wd_stats.append(stats)
wd_df = pd.DataFrame(wd_stats).set_index('星期')
display(wd_df)

print("\n" + "=" * 70)
print("二、出場時間特徵分析")
print("=" * 70)

# ── 2.1 出場15分鐘別績效 ──
print("\n【2.1 出場15分鐘別績效】")
exit_15min_stats = []
for h in sorted(tt['exit_hour'].unique()):
    for m in [0, 15, 30, 45]:
        sub = tt[(tt['exit_hour'] == h) & (tt['exit_minute'] // 15 == m // 15)]['pnl_twd']
        if len(sub) >= 3:  # 樣本數過少的時間段不列入分析
            stats = time_performance_table(sub, f"{h:02d}:{m:02d}")
            stats['出場時間'] = f"{h:02d}:{m:02d}"
            exit_15min_stats.append(stats)
exit_15min_df = pd.DataFrame(exit_15min_stats).set_index('出場時間')
display(exit_15min_df)

# ── 2.2 出場時段（細分）績效 ──
# print("\n【2.2 出場時段（日盤細分）績效】")
# seg_exit_stats = []
# for seg in ['08:45-09:30 開盤初期', '09:30-10:59 早盤', '11:00-11:45 午盤收尾']:
#     sub = tt[tt['exit_session_seg'] == seg]['pnl_twd']
#     if len(sub) > 0:
#         stats = time_performance_table(sub, seg)
#         stats['出場時段'] = seg
#         seg_exit_stats.append(stats)
# seg_exit_df = pd.DataFrame(seg_exit_stats).set_index('出場時段')
# display(seg_exit_df)

print("\n" + "=" * 70)
print("三、交互分析：星期 × 進場小時")
print("=" * 70)

# 建立星期 × 進場小時的 Profit Factor 熱力圖
pf_wd_hr = tt.groupby(['entry_weekday', 'entry_hour'])['pnl_twd'].apply(profit_factor).unstack()
pf_wd_hr.index = [weekday_names.get(i, f"週{i}") for i in pf_wd_hr.index]

# 同時建立樣本數矩陣
cnt_wd_hr = tt.groupby(['entry_weekday', 'entry_hour']).size().unstack(fill_value=0)
cnt_wd_hr.index = [weekday_names.get(i, f"週{i}") for i in cnt_wd_hr.index]

print("\n【3.1 Profit Factor 熱力圖（星期 × 進場小時）】")
print("(數值為每格獨立計算 PF，樣本數過低者僅供參考)")

pf_wd_hr_clip = pf_wd_hr.clip(0, 3)

if HAS_PLOTLY:
    # PF 熱力圖
    fig_pf = go.Figure(go.Heatmap(
        z=pf_wd_hr_clip.values,
        x=[f"{h:02d}:00" for h in pf_wd_hr_clip.columns],
        y=pf_wd_hr_clip.index,
        colorscale='RdYlGn',
        zmid=1.0,
        colorbar=dict(title='PF'),
        text=cnt_wd_hr.values,
        texttemplate="%{text}",
        textfont=dict(size=10),
        hovertemplate='%{y} %{x}<br>PF: %{z:.2f}<br>筆數: %{text}<extra></extra>'
    ))
    fig_pf.update_layout(
        template='plotly_white',
        title='<b>Profit Factor 熱力圖</b>（進場星期 × 進場小時）',
        xaxis_title='進場小時',
        yaxis_title='星期',
        height=350,
    )
    fig_pf.show()

    # 樣本數熱力圖
    fig_cnt = go.Figure(go.Heatmap(
        z=cnt_wd_hr.values,
        x=[f"{h:02d}:00" for h in cnt_wd_hr.columns],
        y=cnt_wd_hr.index,
        colorscale='Blues',
        colorbar=dict(title='筆數'),
        text=cnt_wd_hr.values,
        texttemplate="%{text}",
        textfont=dict(size=10),
    ))
    fig_cnt.update_layout(
        template='plotly_white',
        title='<b>交易筆數</b>（進場星期 × 進場小時）',
        xaxis_title='進場小時',
        yaxis_title='星期',
        height=350,
    )
    fig_cnt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    im1 = axes[0].imshow(pf_wd_hr_clip.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=3)
    axes[0].set_xticks(range(len(pf_wd_hr_clip.columns)))
    axes[0].set_xticklabels([f"{h:02d}:00" for h in pf_wd_hr_clip.columns])
    axes[0].set_yticks(range(len(pf_wd_hr_clip.index)))
    axes[0].set_yticklabels(pf_wd_hr_clip.index)
    axes[0].set_title('Profit Factor（星期 × 進場小時）')
    for i in range(len(pf_wd_hr_clip.index)):
        for j in range(len(pf_wd_hr_clip.columns)):
            axes[0].text(j, i, f"{cnt_wd_hr.iloc[i, j]}", ha="center", va="center", fontsize=8, color="white" if pf_wd_hr_clip.iloc[i, j] < 1.5 else "black")
    plt.colorbar(im1, ax=axes[0], label='PF')

    im2 = axes[1].imshow(cnt_wd_hr.values, aspect='auto', cmap='Blues')
    axes[1].set_xticks(range(len(cnt_wd_hr.columns)))
    axes[1].set_xticklabels([f"{h:02d}:00" for h in cnt_wd_hr.columns])
    axes[1].set_yticks(range(len(cnt_wd_hr.index)))
    axes[1].set_yticklabels(cnt_wd_hr.index)
    axes[1].set_title('交易筆數（星期 × 進場小時）')
    plt.colorbar(im2, ax=axes[1], label='筆數')
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 70)
print("四、月份與季節效應分析")
print("=" * 70)

# ── 4.1 進場月份績效 ──
print("\n【4.1 進場月份績效】")
month_stats = []
for m in range(1, 13):
    sub = tt[tt['entry_month'] == m]['pnl_twd']
    if len(sub) > 0:
        stats = time_performance_table(sub, f"{m}月")
        stats['月份'] = f"{m}月"
        month_stats.append(stats)
month_df = pd.DataFrame(month_stats).set_index('月份')
display(month_df)

# ── 4.2 季節分類 ──
def season(m):
    if m in [3, 4, 5]: return '春季(3-5月)'
    elif m in [6, 7, 8]: return '夏季(6-8月)'
    elif m in [9, 10, 11]: return '秋季(9-11月)'
    else: return '冬季(12-2月)'

tt['entry_season'] = tt['entry_month'].apply(season)
print("\n【4.2 進場季節績效】")
season_stats = []
for s in ['春季(3-5月)', '夏季(6-8月)', '秋季(9-11月)', '冬季(12-2月)']:
    sub = tt[tt['entry_season'] == s]['pnl_twd']
    if len(sub) > 0:
        stats = time_performance_table(sub, s)
        stats['季節'] = s
        season_stats.append(stats)
season_df = pd.DataFrame(season_stats).set_index('季節')
display(season_df)

print("\n" + "=" * 70)
print("五、持有時間與績效關係")
print("=" * 70)

# 持有時間分組
tt['hold_group'] = pd.cut(tt['hold_bars'],
                          bins=[0, 3, 6, 10, 20, 50, 999],
                          labels=['1-3根', '4-6根', '7-10根', '11-20根', '21-50根', '50根+'],
                          include_lowest=True)

print("\n【5.1 持有時間區間績效】")
hold_stats = []
for g in tt['hold_group'].cat.categories:
    sub = tt[tt['hold_group'] == g]['pnl_twd']
    if len(sub) > 0:
        stats = time_performance_table(sub, str(g))
        stats['持有時間'] = str(g)
        hold_stats.append(stats)
hold_df = pd.DataFrame(hold_stats).set_index('持有時間')
display(hold_df)

# 持有時間 vs 損益散點
if HAS_PLOTLY:
    fig_scatter = go.Figure()
    colors = {'多': '#2ecc71', '空': '#e74c3c'}
    for d in tt['direction'].unique():
        sub = tt[tt['direction'] == d]
        fig_scatter.add_trace(go.Scatter(
            x=sub['hold_bars'], y=sub['pnl_twd'],
            mode='markers', name=f'{d}單',
            marker=dict(size=7, opacity=0.6, color=colors.get(d, '#3498db')),
        ))
    fig_scatter.add_hline(y=0, line_dash='dot', line_color='gray')
    fig_scatter.update_layout(
        template='plotly_white',
        title='持有時間 vs 單筆損益散點圖',
        xaxis_title='持有根數（分鐘）',
        yaxis_title='單筆損益 (TWD)',
        height=450,
    )
    fig_scatter.show()
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    for d in tt['direction'].unique():
        sub = tt[tt['direction'] == d]
        ax.scatter(sub['hold_bars'], sub['pnl_twd'], alpha=0.5, label=f'{d}單', s=25)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('持有根數（分鐘）')
    ax.set_ylabel('單筆損益 (TWD)')
    ax.set_title('持有時間 vs 單筆損益散點圖')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 70)
print("六、關鍵發現與優化建議")
print("=" * 70)

# 自動化歸納關鍵發現
def safe_pf(sub):
    gains = sub[sub > 0].sum()
    losses = abs(sub[sub <= 0].sum())
    return gains / losses if losses > 0 else (float('inf') if gains > 0 else 0)

findings = []

# 找出最佳/最差進場小時
pf_by_entry_hr = {h: safe_pf(tt[tt['entry_hour']==h]['pnl_twd']) for h in tt['entry_hour'].unique() if len(tt[tt['entry_hour']==h]) >= 5}
if pf_by_entry_hr:
    best_hr = max(pf_by_entry_hr, key=pf_by_entry_hr.get)
    worst_hr = min(pf_by_entry_hr, key=pf_by_entry_hr.get)
    findings.append(f"• 進場小時：{best_hr:02d}:00 獲利因子最佳 ({pf_by_entry_hr[best_hr]:.2f})，{worst_hr:02d}:00 最差 ({pf_by_entry_hr[worst_hr]:.2f})")

# 時段分析
pf_by_seg = {}
for seg in tt['entry_session_seg'].unique():
    sub = tt[tt['entry_session_seg'] == seg]['pnl_twd']
    if len(sub) >= 5:
        pf_by_seg[seg] = safe_pf(sub)
if pf_by_seg:
    best_seg = max(pf_by_seg, key=pf_by_seg.get)
    worst_seg = min(pf_by_seg, key=pf_by_seg.get)
    findings.append(f"• 進場時段：{best_seg} 表現最佳 (PF={pf_by_seg[best_seg]:.2f})，{worst_seg} 最差 (PF={pf_by_seg[worst_seg]:.2f})")

# 星期分析
pf_by_wd = {wd: safe_pf(tt[tt['entry_weekday']==wd]['pnl_twd']) for wd in range(5) if len(tt[tt['entry_weekday']==wd]) >= 5}
if pf_by_wd:
    best_wd = max(pf_by_wd, key=pf_by_wd.get)
    worst_wd = min(pf_by_wd, key=pf_by_wd.get)
    findings.append(f"• 星期效應：{weekday_names[best_wd]} 表現最佳 (PF={pf_by_wd[best_wd]:.2f})，{weekday_names[worst_wd]} 最差 (PF={pf_by_wd[worst_wd]:.2f})")

# 月份分析
pf_by_month = {m: safe_pf(tt[tt['entry_month']==m]['pnl_twd']) for m in range(1,13) if len(tt[tt['entry_month']==m]) >= 5}
if pf_by_month:
    best_m = max(pf_by_month, key=pf_by_month.get)
    worst_m = min(pf_by_month, key=pf_by_month.get)
    findings.append(f"• 月份效應：{best_m}月 表現最佳 (PF={pf_by_month[best_m]:.2f})，{worst_m}月 最差 (PF={pf_by_month[worst_m]:.2f})")

# 持有時間
pf_by_hold = {}
for g in tt['hold_group'].cat.categories:
    sub = tt[tt['hold_group'] == g]['pnl_twd']
    if len(sub) >= 5:
        pf_by_hold[str(g)] = safe_pf(sub)
if pf_by_hold:
    best_hold = max(pf_by_hold, key=pf_by_hold.get)
    worst_hold = min(pf_by_hold, key=pf_by_hold.get)
    findings.append(f"• 持有時間：{best_hold} 表現最佳 (PF={pf_by_hold[best_hold]:.2f})，{worst_hold} 最差 (PF={pf_by_hold[worst_hold]:.2f})")

print("\n【自動歸納的關鍵發現】")
for f in findings:
    print(f)

print("\n【優化建議方向】")
print("""
1. 時間濾網優化：根據 PF 顯著較差的進場小時/時段，考慮迴避或縮減部位
2. 星期濾網：若特定星期 PF 持續低於 1.0，可考慮當日不交易或降低槓桿
3. 持有時間管理：分析獲利/虧損單的持有時間差異，設定動態出場條件
4. 月份倉控：在歷史績效較差的月份降低風險敞口，績效佳的月份維持或加碼
5. 交互驗證：星期 × 小時熱力圖中，深色（PF < 1）區塊為首要迴避標的
""")

print("\n時間特徵分析完成！")


In [ ]:

# ══════════════════════════════════════════════════════════════════
# Baseline 驗證：ORB 開盤突破策略（git_ignore_folder/baseline_factor.py）
# 用 notebook 的 backtest engine（next_open）跑，和 RD-Agent 結果做比較
# ══════════════════════════════════════════════════════════════════
import datetime
import numpy as np
import pandas as pd

# ── 使用 notebook 已載入的 df ────────────────────────────────────
_df   = df.copy()
open_ = _df['open']; high = _df['high']; low = _df['low']
close = _df['close']; volume = _df['volume']
is_day = (_df['session'] == 'day')
t_arr  = np.array(_df.index.time)
date   = _df.index.normalize()

# Opening range 08:45–09:15
or_mask    = is_day & pd.Series(
    [(x >= datetime.time(8,45)) and (x < datetime.time(9,15)) for x in t_arr],
    index=_df.index)
or_high    = high.where(or_mask).groupby(date).transform('max')
or_low     = low.where(or_mask).groupby(date).transform('min')

# 進場視窗 09:15–13:15；13:30 後強制平倉
active_mask  = is_day & pd.Series(
    [(x >= datetime.time(9,15)) and (x <= datetime.time(13,15)) for x in t_arr],
    index=_df.index)
flatten_mask = is_day & pd.Series(
    [(x >= datetime.time(13,30)) for x in t_arr], index=_df.index)

# VWAP（日盤內累積）
tp       = (high + low + close) / 3.0
tpv      = (tp * volume).where(is_day, 0.0)
vol_d    = volume.where(is_day, 0.0)
vwap     = tpv.groupby(date).cumsum() / vol_d.groupby(date).cumsum().replace(0, np.nan)

# ATR-based position sizing
prev_close = close.shift(1)
tr         = pd.concat([(high-low), (high-prev_close).abs(), (low-prev_close).abs()], axis=1).max(axis=1)
atr20      = tr.rolling(20).mean()
daily_atr  = atr20.where(is_day).groupby(date).mean()
d_atr_med  = daily_atr.rolling(60, min_periods=20).median()
size       = (pd.Series(date.map(d_atr_med), index=_df.index) /
              pd.Series(date.map(daily_atr),  index=_df.index)).clip(0.4, 1.2).fillna(0.0)

# 5-day trend gate
daily_close = close.where(is_day).groupby(date).last()
trend_sign  = pd.Series(
    date.map(np.sign(daily_close.pct_change(5).shift(1))), index=_df.index).fillna(0.0)

# Body + close-position filters
rng      = high - low
body_ok  = (rng > 0) & ((close-open_).abs() / (rng+1e-9) >= 0.5)
cp       = (close - low) / (rng + 1e-9)
cp_long  = (rng > 0) & (cp >= 0.75)
cp_short = (rng > 0) & (cp <= 0.25)

# Entry signals
long_e  = (close > or_high) & active_mask & (close > vwap) & body_ok & cp_long  & (trend_sign >= 0)
short_e = (close < or_low)  & active_mask & (close < vwap) & body_ok & cp_short & (trend_sign <= 0)

raw = pd.Series(0.0, index=_df.index)
raw[long_e]  =  1.0
raw[short_e] = -1.0

# Carry forward within day → flatten at 13:30 → zero night
pos = raw.replace(0.0, np.nan).groupby(date).ffill().fillna(0.0)
pos[flatten_mask] = 0.0
pos[~is_day]      = 0.0

signal_orb = (pos * size).fillna(0.0)

# ── 轉成嚴格 +1/0/-1（notebook engine 需要）──────────────────────
signal_orb_bin = signal_orb.apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

# ── 回測 ─────────────────────────────────────────────────────────
bt_orb  = _df.loc[START_DATE:END_DATE].copy()
sig_orb = signal_orb_bin.loc[START_DATE:END_DATE].copy()

print(f"ORB 信號分布：\n{sig_orb.value_counts().sort_index()}\n")
print(f"非零比例：{(sig_orb != 0).mean()*100:.1f}%")

trades_orb, equity_orb = run_backtest(bt_orb, sig_orb, cfg)
print()
print_report(trades_orb, equity_orb, cfg)


## Baseline 驗證：ORB 開盤突破策略（baseline_factor.py）

比較 RD-Agent 回測引擎 vs notebook 回測引擎，使用相同策略邏輯。


## Baseline 驗證：ORB 開盤突破策略（baseline_factor.py）

用 notebook 的 backtest engine (next_open) 跑，與 RD-Agent 結果比較。

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Baseline 驗證：ORB 開盤突破策略
# 來源：git_ignore_folder/baseline_factor.py
# 回測引擎：backtest_engiene.py（next_open），和 RD-Agent vectorized 做比較
# ══════════════════════════════════════════════════════════════════
import datetime
import numpy as np
import pandas as pd

_df   = df.copy()
open_ = _df['open']; high = _df['high']; low = _df['low']
close = _df['close']; volume = _df['volume']
is_day = (_df['session'] == 'day')
t_arr  = np.array(_df.index.time)
date   = _df.index.normalize()

# Opening range 08:45–09:15
or_mask = is_day & pd.Series(
    [(x >= datetime.time(8,45)) and (x < datetime.time(9,15)) for x in t_arr],
    index=_df.index)
or_high = high.where(or_mask).groupby(date).transform('max')
or_low  = low.where(or_mask).groupby(date).transform('min')

# 進場視窗 09:15–13:15；13:30 後強制平倉
active_mask  = is_day & pd.Series(
    [(x >= datetime.time(9,15)) and (x <= datetime.time(13,15)) for x in t_arr],
    index=_df.index)
flatten_mask = is_day & pd.Series(
    [(x >= datetime.time(13,30)) for x in t_arr], index=_df.index)

# VWAP（日盤內累積）
tp   = (high + low + close) / 3.0
tpv  = (tp * volume).where(is_day, 0.0)
vol_d = volume.where(is_day, 0.0)
vwap = tpv.groupby(date).cumsum() / vol_d.groupby(date).cumsum().replace(0, np.nan)

# ATR position sizing
prev_close = close.shift(1)
tr    = pd.concat([(high-low),(high-prev_close).abs(),(low-prev_close).abs()],axis=1).max(axis=1)
atr20 = tr.rolling(20).mean()
daily_atr    = atr20.where(is_day).groupby(date).mean()
d_atr_med    = daily_atr.rolling(60, min_periods=20).median()
size = (pd.Series(date.map(d_atr_med), index=_df.index) /
        pd.Series(date.map(daily_atr),  index=_df.index)).clip(0.4, 1.2).fillna(0.0)

# 5-day trend gate
daily_close = close.where(is_day).groupby(date).last()
trend_sign  = pd.Series(
    date.map(np.sign(daily_close.pct_change(5).shift(1))), index=_df.index).fillna(0.0)

# Body + close-position filters
rng      = high - low
body_ok  = (rng > 0) & ((close-open_).abs() / (rng+1e-9) >= 0.5)
cp       = (close - low) / (rng + 1e-9)
cp_long  = (rng > 0) & (cp >= 0.75)
cp_short = (rng > 0) & (cp <= 0.25)

# Entries
long_e  = (close > or_high) & active_mask & (close > vwap) & body_ok & cp_long  & (trend_sign >= 0)
short_e = (close < or_low)  & active_mask & (close < vwap) & body_ok & cp_short & (trend_sign <= 0)
raw = pd.Series(0.0, index=_df.index)
raw[long_e] = 1.0; raw[short_e] = -1.0

# Carry within day, flatten 13:30+, zero night
pos = raw.replace(0.0, np.nan).groupby(date).ffill().fillna(0.0)
pos[flatten_mask] = 0.0
pos[~is_day] = 0.0

signal_orb = (pos * size).fillna(0.0)
signal_orb_bin = signal_orb.apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))

print("ORB 信號分布：")
print(signal_orb_bin.value_counts().sort_index())
print(f"\n在市場比例：{(signal_orb_bin != 0).mean()*100:.1f}%")

# 回測
bt_orb  = _df.loc[START_DATE:END_DATE].copy()
sig_orb = signal_orb_bin.loc[START_DATE:END_DATE].copy()

trades_orb, equity_orb = run_backtest(bt_orb, sig_orb, cfg)
print()
print_report(trades_orb, equity_orb, cfg)
